# Find out the dimension of interaction blocks between 1st and 2nd layer

In [ ]:
import torch
from mace import data, modules, tools
import numpy as np
import torch.nn.functional
from e3nn import o3
import ase.io
import matplotlib.pyplot as plt
import warnings
import copy
import pandas as pd
warnings.filterwarnings("ignore")
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
import openequivariance

import ns_plots


try:
    import cuequivariance as cue
    cueq_available = True
    print("✓ cuEquivariance library is available")
except ImportError:
    cueq_available = False
    print("✗ cuEquivariance library is not available - cuEq will be disabled")

/homes/ab3149/miniconda3/envs/myvenv/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch.l

✓ cuEquivariance library is available


In [2]:
def get_default_model_config(z_table):
    # setup some default parameters based on the actual dataset
    num_elements = len(z_table.zs)
    # Create atomic energies array with default values for each element
    # You can adjust these values based on your needs
    atomic_energies = np.array([-1.0] * num_elements, dtype=float)  # Default energy per element
    cutoff = 3

    default_model_config = dict(
            num_elements=num_elements,  # number of chemical elements (dynamic)
            atomic_energies=atomic_energies,  # atomic energies used for normalisation
            avg_num_neighbors=8,  # avg number of neighbours of the atoms, used for internal normalisation of messages
            atomic_numbers=z_table.zs,  # atomic numbers, used to specify chemical element embeddings of the model
            r_max=cutoff,  # cutoff
            num_bessel=8,  # number of radial features
            num_polynomial_cutoff=6,  # smoothness of the radial cutoff
            max_ell=2,  # expansion order of spherical harmonic adge attributes
            num_interactions=2,  # number of layers, typically 2
            interaction_cls_first=modules.interaction_classes["RealAgnosticInteractionBlock"],
            interaction_cls=modules.interaction_classes["RealAgnosticInteractionBlock"],
            hidden_irreps=o3.Irreps("8x0e + 8x1o"),  # 8: number of embedding channels, 0e, 1o is specifying which equivariant messages to use. Here up to L_max=1
            correlation=3,  # correlation order of the messages (body order - 1)
            MLP_irreps=o3.Irreps("16x0e"),  # number of hidden dimensions of last layer readout MLP
            gate=torch.nn.functional.silu,  # nonlinearity used in last layer readout MLP
        )

    return default_model_config


def data_prep():
    single_molecule = ase.io.read('md22_double-walled_nanotube.xyz', index='0')

    # Detect elements present in the dataset
    atomic_numbers = single_molecule.numbers
    unique_atomic_numbers = sorted(set(atomic_numbers))
    print(f"Elements found in dataset: {unique_atomic_numbers}")
    print(f"Element symbols: {single_molecule.get_chemical_symbols()[:10]}...")  # Show first 10 symbols
    
    Rcut = 3.0 # cutoff radius
    # z_table = tools.AtomicNumberTable([1, 6, 8])
    z_table = tools.AtomicNumberTable(unique_atomic_numbers)
    print(f"Created z_table with {len(z_table.zs)} elements: {z_table.zs}")

    config = data.Configuration(
        atomic_numbers=single_molecule.numbers,
        positions=single_molecule.positions,
        properties={},
        property_weights={},
    )

    # we handle configurations using the AtomicData class
    batch = data.AtomicData.from_config(config, z_table=z_table, cutoff=Rcut)

    vectors, lengths = modules.utils.get_edge_vectors_and_lengths(
    positions=batch["positions"],
    edge_index=batch["edge_index"],
    shifts=batch["shifts"],
    )
    print(f'there are {batch.positions.shape[0]} nodes and {len(lengths)} edges')
    print(f'lengths is shape {lengths.shape}')
    print(f'vectors is shape {vectors.shape}')

    return batch, lengths, vectors, z_table

In [6]:
# -----------------------------------------------------------
# 1. helper to guarantee ptr & batch, and drop head
# -----------------------------------------------------------
def ensure_ptr_batch_and_no_head(data_dict: dict) -> dict:
    """
    - If 'ptr' missing or None: assume whole thing is one graph, create ptr=[0,N].
    - If 'batch' missing or None: create it via ptr so every atom gets a graph index.
    - If 'head' exists: DROP it, so prepare_graph uses its default zero vector.
    """
    N = int(data_dict["positions"].shape[0])

    # drop any stray head
    data_dict.pop("head", None)

    # ptr
    if "ptr" not in data_dict or data_dict["ptr"] is None:
        data_dict["ptr"] = torch.tensor([0, N], dtype=torch.long,
                                        device=data_dict["positions"].device)

    # batch
    if "batch" not in data_dict or data_dict["batch"] is None:
        ptr = data_dict["ptr"]
        counts = torch.diff(ptr)  # number of atoms per graph
        graph_ids = torch.arange(len(counts), device=ptr.device)
        data_dict["batch"] = torch.repeat_interleave(graph_ids, counts)

    return data_dict


# -----------------------------------------------------------
# 2. hook printers (unchanged)
# -----------------------------------------------------------
from functools import partial

def _print_io(label, module, inp, out):
    in_shapes  = [tuple(t.shape) for t in inp]
    out_shapes = [tuple(o.shape) for o in (out if isinstance(out, tuple) else (out,))]
    print(f"{label:<30s}: {in_shapes}  ->  {out_shapes}")

def _print_pre(label, module, inp):
    in_shapes = [tuple(t.shape) for t in inp]
    print(f"{label:<30s}: (pre) {in_shapes}")

def add_shape_hooks(interaction, idx):
    want = {
        "linear_up",        # first linear 
        "conv_tp_weights",  # radial MLP
        "conv_tp",          # TP message
        "linear",           # post-aggregate linear
    }
    for name, sub in interaction.named_modules():
        if name in want:
            # show pre-aggregate on 'linear'
            if name == "linear":
                sub.register_forward_pre_hook(
                    partial(_print_pre, f"[L{idx}] {name}")
                )
            sub.register_forward_hook(
                partial(_print_io, f"[L{idx}] {name}")
            )


# -----------------------------------------------------------
# 3. full trace function
# -----------------------------------------------------------
def trace_shapes(device="cpu"):
    # ---- prepare data ----
    batch, lengths, vectors, z_table = data_prep()  # your data_prep
    if hasattr(batch, "to_dict"):                    # PyG Batch?
        batch = batch.to_dict()
    batch = ensure_ptr_batch_and_no_head(batch)

    # ---- build model ----
    torch.set_default_dtype(torch.float64)
    cfg   = get_default_model_config(z_table)        # your config fn
    model = modules.MACE(**cfg).to(device=device,
                                   dtype=torch.float64)

    # ---- list sub-modules once ----
    print("\n--- interaction[0] sub-modules ---")
    for n, _ in model.interactions[0].named_modules():
        print(" ", n)
    print("-------------------------------------\n")

    # ---- attach hooks & run ----
    add_shape_hooks(model.interactions[0], idx=0)
    add_shape_hooks(model.interactions[1], idx=1)
    _ = model(batch)  # prints all the shapes


# finally, run it
trace_shapes(device="cpu")

Elements found in dataset: [np.int64(1), np.int64(6)]
Element symbols: ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C']...
Created z_table with 2 elements: [np.int64(1), np.int64(6)]
there are 370 nodes and 4270 edges
lengths is shape torch.Size([4270, 1])
vectors is shape torch.Size([4270, 3])

--- interaction[0] sub-modules ---
  
  linear_up
  linear_up._compiled_main
  conv_tp
  conv_tp._compiled_main_left_right
  conv_tp._compiled_main_right
  conv_tp_weights
  conv_tp_weights.layer0
  conv_tp_weights.layer0.act
  conv_tp_weights.layer1
  conv_tp_weights.layer2
  conv_tp_weights.layer3
  linear
  linear._compiled_main
  skip_tp
  skip_tp._compiled_main_left_right
  skip_tp._compiled_main_right
  reshape
-------------------------------------

[L0] linear_up                : [(370, 8)]  ->  [(370, 8)]
[L0] conv_tp_weights          : [(4270, 8)]  ->  [(4270, 24)]
[L0] conv_tp                  : [(4270, 8), (4270, 9), (4270, 24)]  ->  [(4270, 72)]
[L0] linear                   : (pr

---

## This test the difference of computational cost between layer 0 and layer 2

In [3]:
import types

def with_cueq_conv_fusion(conv_tp: torch.nn.Module) -> torch.nn.Module:
    """Wraps a cuet.ConvTensorProduct to use conv fusion"""
    conv_tp.original_forward = conv_tp.forward

    def forward(
        self,
        node_feats: torch.Tensor,
        edge_attrs: torch.Tensor,
        tp_weights: torch.Tensor,
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        sender = edge_index[0]
        receiver = edge_index[1]
        return self.original_forward(
            [tp_weights, node_feats, edge_attrs],
            {1: sender},
            {0: node_feats},
            {0: receiver},
        )

    conv_tp.forward = types.MethodType(forward, conv_tp)
    return conv_tp

In [4]:
def get_memory_and_time_usage(device: str, task_name: callable):
    if device.startswith('cuda'):

        torch.cuda.reset_peak_memory_stats(device)
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        task_name
        end.record()
        torch.cuda.synchronize()
        t = start.elapsed_time(end)
        m = torch.cuda.max_memory_allocated(device)
        return t, m
    else:
        import time
        t = time.perf_counter()
        task_name()
        t = (time.perf_counter() - t) * 1000
        m = 0
        return t, m

In [ ]:
import torch, copy
from e3nn.o3 import Irreps
import cuequivariance   as cue
import cuequivariance_torch as cuet

def benchmark_cuda(fn, warmup=10, runs=50):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(runs):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end)/runs, torch.cuda.max_memory_allocated()

def benchmark_cpu(fn, warmup=3, runs=10):
    import time
    for _ in range(warmup):
        fn()
    t0 = time.perf_counter()
    for _ in range(runs):
        fn()
    return (time.perf_counter()-t0)*1000/runs, 0

def get_memory_and_time_usage(device: str, fn):
    return (benchmark_cuda if device.startswith("cuda") else benchmark_cpu)(fn)


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
torch.manual_seed(0)

batch, lengths, vectors, z_table = data_prep()
cfg   = get_default_model_config(z_table)
model = modules.MACE(**cfg).to(device=device, dtype=torch.float64)

# grab the two TensorProduct modules because we want to get the dimensions of the weights
tp0 = model.interactions[0].conv_tp # first layer
tp1 = model.interactions[1].conv_tp # second layer

# turn their irreps into e3nn.Irreps to get dimensions
in_ir0, attr_ir0, out_ir0 = (
    Irreps(tp0.irreps_in1),
    Irreps(tp0.irreps_in2),
    Irreps(tp0.irreps_out),
)
in_ir1, attr_ir1, out_ir1 = (
    Irreps(tp1.irreps_in1),
    Irreps(tp1.irreps_in2),
    Irreps(tp1.irreps_out),
)

# sanity check printing

print("L0 output dim:", out_ir0.dim)
print("L1 output dim:", out_ir1.dim)

print("Layer 0 conv_tp dims:")
print(" - in feat dim        ", in_ir0.dim)
print(" - edge-attr (Y_l) dim ", attr_ir0.dim)
print(" - radial-MLP weight dim", tp0.weight_numel)
print(" - out feat dim       ", out_ir0.dim, "\n")

print("Layer 1 conv_tp dims:")
print(" - in feat dim        ", in_ir1.dim)
print(" - edge-attr (Y_l) dim ", attr_ir1.dim)
print(" - radial-MLP weight dim", tp1.weight_numel)
print(" - out feat dim       ", out_ir1.dim, "\n")

# TODO: build the polynomial module with the target compute dtype:

# build cue descriptors + polynomial modules which allows us to use the cuequivariance library and directly conv_tp without
# having to write our own custom conv_tp config file which is annoying atm
def make_poly(in_ir, attr_ir, out_ir):
    desc = cue.descriptors.channelwise_tensor_product(
        cue.Irreps("O3", in_ir),
        cue.Irreps("O3", attr_ir),
        cue.Irreps("O3", out_ir),
    )
    return cuet.SegmentedPolynomial(
        desc.flatten_coefficient_modes()
            .squeeze_modes()
            .polynomial,
        math_dtype=torch.float64,
        output_dtype_map=[-1]
    ).to(device).double()

# we are building for the first layer and second layer
poly0 = make_poly(in_ir0, attr_ir0, out_ir0)
poly1 = make_poly(in_ir1, attr_ir1, out_ir1)

# fuse them with the helper function which is given by MACE in wrapper_ops.py
fused0_ref = with_cueq_conv_fusion(poly0)
fused1_ref = with_cueq_conv_fusion(poly1)

# cast layers once and store dtype
layers0, layers1 = {}, {}
for name, dtype in [
    ("FP64", torch.float64),
    ("FP32", torch.float32),
    ("TF32","tf32"),
    ("FP16", torch.float16),
    ("BF16", torch.bfloat16),
]:
    if name in ("TF32","FP16","BF16") and not device.startswith("cuda"):
        continue
    if name == "TF32" and device.startswith("cuda"):
        dtype = torch.float32
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32   = True

    m0 = copy.deepcopy(fused0_ref).to(dtype)
    m1 = copy.deepcopy(fused1_ref).to(dtype)
    layers0[name] = (m0, dtype)
    layers1[name] = (m1, dtype)

# simulate a mini-batch with the same number of nodes and edges as the real run
N, E = 200000, 2000000
nf0 = torch.randn(N, in_ir0.dim,    dtype=torch.float64)
nf1 = torch.randn(N, in_ir1.dim,    dtype=torch.float64)
ea  = torch.randn(E, attr_ir0.dim,  dtype=torch.float64)
tw0 = torch.randn(E, tp0.weight_numel, dtype=torch.float64)
tw1 = torch.randn(E, tp1.weight_numel, dtype=torch.float64)
ei  = torch.randint(0, N, (2, E), dtype=torch.int64)

inputs0, inputs1 = {}, {}
for name, (layer0, dt0) in layers0.items():
    inputs0[name] = (
        nf0.to(device=device, dtype=dt0),
        ea .to(device=device, dtype=dt0),
        tw0.to(device=device, dtype=dt0),
        ei .to(device)
    )
    layer1, dt1 = layers1[name]
    inputs1[name] = (
        nf1.to(device=device, dtype=dt1),
        ea .to(device=device, dtype=dt1),
        tw1.to(device=device, dtype=dt1),
        ei .to(device)
    )

# benchmark forward only
results = []
for name in layers0:
    layer0, _ = layers0[name]
    layer1, _ = layers1[name]

    task0 = lambda: layer0(*inputs0[name])[0]
    task1 = lambda: layer1(*inputs1[name])[0]

    t0, m0 = get_memory_and_time_usage(device, task0)
    t1, m1 = get_memory_and_time_usage(device, task1)
    results.append((name, t0, m0, t1, m1))

# print a comparative table for memory and executiong time 
# tensor comparison between different dtypes one each layer done in another notebook
print(f"{'dtype':<6s}  L0_time  L0_mem   L1_time  L1_mem   L1/L0_time  L1/L0_mem")
for name, t0, m0, t1, m1 in results:
    print(f"{name:<6s} {t0:.3f}ms {m0/1e6:.3f}MB {t1:.3f}ms {m1/1e6:.3f}MB "
          f"{t1/t0:.3f}×     {m1/m0:.3f}×")


Using device: cuda
Elements found in dataset: [np.int64(1), np.int64(6)]
Element symbols: ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C']...
Created z_table with 2 elements: [np.int64(1), np.int64(6)]
there are 370 nodes and 4270 edges
lengths is shape torch.Size([4270, 1])
vectors is shape torch.Size([4270, 3])
L0 output dim: 72
L1 output dim: 168
Layer 0 conv_tp dims:
 - in feat dim         8
 - edge-attr (Y_l) dim  9
 - radial-MLP weight dim 24
 - out feat dim        72 

Layer 1 conv_tp dims:
 - in feat dim         32
 - edge-attr (Y_l) dim  9
 - radial-MLP weight dim 56
 - out feat dim        168 

dtype   L0_time  L0_mem   L1_time  L1_mem   L1/L0_time  L1/L0_mem
FP64   11.117ms 4519.794MB 25.535ms 4673.394MB 2.297×     1.034×
FP32   9.255ms 4519.794MB 30.719ms 4673.394MB 3.319×     1.034×
TF32   9.255ms 4519.794MB 30.720ms 4673.394MB 3.319×     1.034×
FP16   14.483ms 4519.794MB 31.019ms 4673.394MB 2.142×     1.034×
BF16   9.207ms 4519.794MB 37.447ms 4673.394MB 4.067×     1.034